In [5]:
import os
import gc
import zarr
import yaml
import json
import numba
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm
import statsmodels.api as sm

from plotnine import *
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'

In [6]:
# Configuration and paths
eur_samples_path = '/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv'

maf=1e-3
n = !wc -l $eur_samples_path
n_eur = int(n[0].split(' ')[0])
mac = maf*(2*n_eur)

config_path = "/home/dnanexus/ukbgym/config_wgs.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

# Read the variant consequence configuration
records = []
for group, consequences in config["variant_consequences"].items():
    for consequence in consequences:
        records.append({'variant_class': group, 'vep_consequence': consequence})
var_cons = pl.DataFrame(records)


# Use a list comprehension to flatten the nested dictionary into records
records = [
    {
        "category": category,
        "annotation": anno,
        "color": props["color"],
        "label": props["label"],
        "annotation_dir": props.get("direction", 1),
    }
    for category, annos in config["rare_variant_annotations"].items()
    for anno, props in annos.items()
]

# Create the DataFrame directly from the list of records
anno_config_df = pl.DataFrame(records)
all_annotation_list = anno_config_df.select(pl.col("annotation")).to_series().to_list()

anno_config_df

category,annotation,color,label,annotation_dir
str,str,str,str,i64
"""plof""","""loftee_hc""","""#DD4344""","""LOFTEE HC""",1
"""missense""","""am_pathogenicity""","""#feb72d""","""AlphaMissense""",1
"""missense""","""score_pai3d""","""#feb72d""","""PrimateAI-3D""",1
"""missense""","""esmscoremissense""","""#feb72d""","""ESM1v""",-1
"""genetic_diversity""","""cadd_raw""","""#1f77b4""","""CADD Raw""",1
…,…,…,…,…
"""splicing""","""absplice_dna_max""","""#28a745""","""AbSplice (max)""",1
"""splicing""","""absplice2_max""","""#28a745""","""AbSplice2 (max)""",1
"""regulatory_nondir""","""promoterai_abs""","""#00A99D""","""PromoterAI abs""",1


In [7]:
exp_annos_df = pl.DataFrame({
    "category": ['non-coding indel', 'non-coding indel', 'non-coding indel', 'non-coding indel'],
    'annotation': ['noncoding_indel', 'intronic_indel', 'utr5_indel', 'utr3_indel'],
    "color": ['gray', 'gray', 'gray', 'gray'],
    'label': ['non-coding indel', 'non-coding indel', 'non-coding indel', 'non-coding indel'],
    'annotation_dir': [1, 1, 1, 1],
})

anno_config_df = pl.concat([anno_config_df, exp_annos_df])

In [8]:
anno = pl.scan_parquet("/home/dnanexus/data_dir/genebass394genes_olink371genes_annotated_MANE_CADD_ENCODE_251117.parquet")
anno.head().collect()

id,region,gene_name,chrom,pos,ref,alt,qual,filter,info,location,allele,gene,feature,feature_type,consequence,cdna_position,cds_position,protein_position,amino_acids,codons,existing_variation,impact,distance,strand,flags,biotype,canonical,ensp,sift,polyphen,gnomade_af,gnomade_afr_af,gnomade_amr_af,gnomade_asj_af,gnomade_eas_af,gnomade_fin_af,…,cpg,oaa,naa,ccds,intron,exon,cdnapos,relcdnapos,cdspos,relcdspos,protpos,relprotpos,polyphencat,polyphenval,priphcons,mamphcons,verphcons,priphylop,mamphylop,verphylop,ensembleregulatoryfeature,esmscoremissense,esmscoreinframe,esmscoreframeshift,aparent2,zoopriphylop,zooverphylop,tss,strand_right,gene_length,dist_to_tss,pangolin_score,delta_score,absplice_dna_max,absplice2_max,encode_feature,encode_feature_name
str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,str,str,str,str,str,f32,f32,f32,f32,f32,f32,str,f32,f32,f32,f32,f32,f32,f32,str,f32,f32,f32,f32,f32,f32,i64,cat,i64,i64,f64,f64,f64,f64,str,str
"""chr1:158629580:T:C""","""ENSG00000163554""","""SPTA1""","""chr1""",158629580,"""T""","""C""",""".""",""".""",""".""","""chr1:158629580""","""C""","""ENSG00000163554""","""ENST00000643759""","""Transcript""","""intron_variant""",null,null,null,null,null,"""rs749252938""","""MODIFIER""",null,"""-1""",null,"""protein_coding""","""YES""","""ENSP00000495214""",null,null,null,null,null,null,null,null,…,0.027,null,null,"""CCDS41423.1""","""39/51""",null,null,null,null,null,null,null,null,null,0.029,0.0,0.0,0.22,-1.301,-1.395,null,null,null,null,null,0.001,-1.413,158686715,"""-""",76013,57135,0.0,0.0,0.001,0.000033,"""Not annotated in ENCODE""","""-1"""
"""chr1:158618272:G:A""","""ENSG00000163554""","""SPTA1""","""chr1""",158618272,"""G""","""A""",""".""",""".""",""".""","""chr1:158618272""","""A""","""ENSG00000163554""","""ENST00000643759""","""Transcript""","""intron_variant""",null,null,null,null,null,null,"""MODIFIER""",null,"""-1""",null,"""protein_coding""","""YES""","""ENSP00000495214""",null,null,null,null,null,null,null,null,…,0.0,null,null,"""CCDS41423.1""","""45/51""",null,null,null,null,null,null,null,null,null,0.177,0.002,0.002,0.595,1.346,1.209,null,null,null,null,null,0.002,0.471,158686715,"""-""",76013,68443,0.0,0.0,0.001,0.000033,"""CA-CTCF""","""EH38E1389453"""
"""chr1:158618428:C:T""","""ENSG00000163554""","""SPTA1""","""chr1""",158618428,"""C""","""T""",""".""",""".""",""".""","""chr1:158618428""","""T""","""ENSG00000163554""","""ENST00000643759""","""Transcript""","""intron_variant""",null,null,null,null,null,"""rs942206589""","""MODIFIER""",null,"""-1""",null,"""protein_coding""","""YES""","""ENSP00000495214""",null,null,null,null,null,null,null,null,…,0.013,null,null,"""CCDS41423.1""","""45/51""",null,null,null,null,null,null,null,null,null,0.078,0.0,0.0,0.418,-0.322,-0.297,null,null,null,null,null,0.002,0.053,158686715,"""-""",76013,68287,0.0,0.0,0.001,0.000033,"""Not annotated in ENCODE""","""-1"""
"""chr1:158622449:T:G""","""ENSG00000163554""","""SPTA1""","""chr1""",158622449,"""T""","""G""",""".""",""".""",""".""","""chr1:158622449""","""G""","""ENSG00000163554""","""ENST00000643759""","""Transcript""","""intron_variant""",null,null,null,null,null,null,"""MODIFIER""",null,"""-1""",null,"""protein_coding""","""YES""","""ENSP00000495214""",null,null,null,null,null,null,null,null,…,0.0,null,null,"""CCDS41423.1""","""43/51""",null,null,null,null,null,null,null,null,null,0.006,0.0,0.0,0.316,0.306,0.323,null,null,null,null,null,0.001,0.503,158686715,"""-""",76013,64266,0.0,0.0,0.001,0.000033,"""Not annotated in ENCODE""","""-1"""
"""chr1:158624418:C:T""","""ENSG00000163554""","""SPTA1""","""chr1""",158624418,"""C""","""T""",""".""",""".""",""".""","""chr1:158624418""","""T""","""ENSG00000163554""","""ENST00000643759""","""Transcript""","""intron_variant""",null,null,null,null,null,null,"""MODIFIER""",null,"""-1""",null,"""protein_coding""","""YES""","""ENSP00000495214""",nu

In [ ]:
# anno = pl.scan_parquet("/home/dnanexus/data_dir/genebass394genes_olink371genes_variants_union_annotated_genocode_251103.parquet")
anno = pl.scan_parquet("/home/dnanexus/data_dir/genebass394genes_olink371genes_annotated_MANE_CADD_ENCODE_251117.parquet")

# existing_annos = [c for c in all_annotation_list if c in anno.collect_schema().names()]

# existing_annos = ['loftee_hc', 'am_pathogenicity']#, 'noncoding_indel', 'intronic_indel', 'utr5_indel', 'utr3_indel', 'insertion_1bp', 'deletion_1bp', 'insertion_2_10bp', 'deletion_2_10bp', 'insertion_10bp', 'deletion_10bp']

# Filter variant classes
variant_class = "coding"
consequences_for_group = var_cons.filter(
    pl.col('variant_class') == variant_class
)['vep_consequence'].to_list()
filter_expression = pl.any_horizontal(
    (pl.col(c) == 1) for c in consequences_for_group if c in anno.collect_schema().names()
)

min_range = -2000
max_range = +0

anno = (
    anno
    .filter(
        # (pl.col('vep_cds') == True)
        # ((pl.col('vep_cds') == True) | (pl.col('mane_cds') == True)) &
        # (pl.col('non_mane_cds') == False)

        (pl.col('dist_to_tss') < 0)
    )
    
    # .with_columns(
    #     intronic_indel = pl.col('noncoding_indel') & (pl.col('consequence_intron_variant')==1),
    #     utr5_indel = pl.col('noncoding_indel') & (pl.col('consequence_5_prime_utr_variant')==1),
    #     utr3_indel = pl.col('noncoding_indel') & (pl.col('consequence_3_prime_utr_variant')==1),

    #     insertion_1bp = (pl.col('indel_length')==1) & (pl.col('noncoding_insertion')==True),
    #     deletion_1bp = (pl.col('indel_length')==1) & (pl.col('noncoding_deletion')==True),
    #     insertion_2_10bp = (pl.col('indel_length')>1) & (pl.col('indel_length')<10) & (pl.col('noncoding_insertion')==True),
    #     deletion_2_10bp = (pl.col('indel_length')>1) & (pl.col('indel_length')<10) & (pl.col('noncoding_deletion')==True),
    #     insertion_10bp = (pl.col('indel_length')>10) & (pl.col('noncoding_insertion')==True),
    #     deletion_10bp = (pl.col('indel_length')>10) & (pl.col('noncoding_deletion')==True),
    # )
    .select(
        set(['id', 'region', 'tss', 'strand', 'gene_length', 'gene_name', 'dist_to_tss']).union(set(existing_annos))
    )
    .collect(engine='streaming')
    # .drop_nulls()
)

anno

gene_name,id,loftee_hc,strand,gene_length,dist_to_tss,tss,region,am_pathogenicity
str,str,i8,str,i64,i64,i64,str,f32
"""CHD2""","""chr15:92894026:G:A""",0,"""1""",127809,-6162,92900188,"""ENSG00000173575""",null
"""CHD2""","""chr15:92883206:C:T""",0,"""1""",127809,-16982,92900188,"""ENSG00000173575""",null
"""CHD2""","""chr15:92885997:A:G""",0,"""1""",127809,-14191,92900188,"""ENSG00000173575""",null
"""CHD2""","""chr15:92890145:G:A""",0,"""1""",127809,-10043,92900188,"""ENSG00000173575""",null
"""CHD2""","""chr15:92883449:C:G""",0,"""1""",127809,-16739,92900188,"""ENSG00000173575""",null
…,…,…,…,…,…,…,…,…
"""RMC1""","""chr18:23499426:A:ACAC""",0,"""1""",28354,-4043,23503469,"""ENSG00000141452""",null
"""SH2B3""","""chr12:111404754:G:GC""",0,"""1""",45702,-1168,111405922,"""ENSG00000111252""",null
"""R3HDM4""","""chr19:917933:A:AGCGCGGGCGG""",0,"""-1""",16744,-4688,913245,"""ENSG00000198858""",null


In [58]:
melted_anno = (
    anno
    .with_columns(
        am_pathogenicity = pl.col('am_pathogenicity') >= 0.8
    )

    .unpivot(
        index=["id", "region"],
        on=existing_annos,
        variable_name="annotation",
        value_name="annotation_score"
    )
    .with_columns(
        pl.col("annotation_score").cast(pl.Float32),
        pl.col("region").cast(pl.Utf8),
    )

    # Merge with annotation configuration to get direction and filter
    .join(
        anno_config_df,
        on="annotation",
        how="left"
    )
    .with_columns(
        annotation_score_dircor = pl.col('annotation_score') * pl.col("annotation_dir")
    )

    # .filter(pl.col('annotation_score').is_not_null())
)

melted_anno

id,region,annotation,annotation_score,category,color,label,annotation_dir,annotation_score_dircor
str,str,str,f32,str,str,str,i64,f64
"""chr1:158647693:G:A""","""ENSG00000163554""","""loftee_hc""",0.0,"""plof""","""#DD4344""","""LOFTEE HC""",1,0.0
"""chr1:158612895:C:G""","""ENSG00000163554""","""loftee_hc""",0.0,"""plof""","""#DD4344""","""LOFTEE HC""",1,0.0
"""chr1:158613759:G:A""","""ENSG00000163554""","""loftee_hc""",0.0,"""plof""","""#DD4344""","""LOFTEE HC""",1,0.0
"""chr1:158618054:G:T""","""ENSG00000163554""","""loftee_hc""",0.0,"""plof""","""#DD4344""","""LOFTEE HC""",1,0.0
"""chr1:158635989:C:T""","""ENSG00000163554""","""loftee_hc""",0.0,"""plof""","""#DD4344""","""LOFTEE HC""",1,0.0
…,…,…,…,…,…,…,…,…
"""chr7:6692051:G:GT""","""ENSG00000164631""","""am_pathogenicity""",null,"""missense""","""#feb72d""","""AlphaMissense""",1,null
"""chr2:151435556:AATC:A""","""ENSG00000080345""","""am_pathogenicity""",null,"""missense""","""#feb72d""","""AlphaMissense""",1,null
"""chr19:3982946:TTCA:T""","""ENSG00000167658""","""am_pathogenicity""",null,"""missense""","""#feb72d""","""AlphaMissense""",1,null


In [59]:
appv = pl.scan_parquet("/home/dnanexus/data_dir/appv_files/avg_pheno_per_var_quantitative_EUR_genebass1e6_PRScorr_with_percentiles.parquet")

# Create a lazy frame with the unique keys
anno_keys = anno.select(pl.col('id').unique()).lazy()

# Chain the filter and the much faster semi join
appv = (
    appv
        .join(
        anno_keys, on='id', how='semi'
    )
    .filter(
        pl.col('n_individuals') <= mac
    )
    .select(
        ['id', 'phenotype', 'mean_pheno_value_ptile', 'n_individuals']
    )
)

# unique_phenotypes = appv.select('phenotype').unique().collect(engine='streaming').to_series()

In [60]:
# Get gene trait associations

plof = pl.read_parquet('/home/dnanexus/data_dir/association_files/rvat_EUR_500k_regenie.parquet').with_columns(
    phenotype = (pl.col('trait') + '_int'),
    region = pl.col('gene_id'),
    rvat_pval = (10** -pl.col("neg_log10p")),
).filter(
    (pl.col('trait_type') == 'quantitative')
)

loftee_corr = pl.read_parquet("/home/dnanexus/data_dir/association_files/loftee_correlation_quantitative_wgs_EUR_genebass1e6_maf1e3_snp_consistent.parquet")

gene_trait_df = loftee_corr.join(plof[['region', 'phenotype', 'beta', 'rvat_pval']], on=['region', 'phenotype'], how='inner').filter(
    pl.col('loftee_corr')*pl.col('beta') > 0
).with_columns(
    corr_dir = pl.col('loftee_corr')/pl.col('loftee_corr').abs()
)

# gene_trait_df = gene_trait_df.head()
gene_trait_df

region,gene_name,phenotype,loftee_corr,n_variants,beta,rvat_pval,corr_dir
str,str,str,f64,u64,f64,f64,f64
"""ENSG00000116183""","""PAPPA2""","""arm_fatfree_mass_right_int""",-0.014747,82603,-0.195409,9.1637e-8,-1.0
"""ENSG00000129083""","""COPB1""","""arm_fatfree_mass_right_int""",-0.014338,13594,-0.397279,0.006626,-1.0
"""ENSG00000100578""","""KIAA0586""","""arm_fatfree_mass_right_int""",-0.010787,28541,-0.058463,0.000305,-1.0
"""ENSG00000157766""","""ACAN""","""arm_fatfree_mass_right_int""",-0.036259,20244,-0.436618,6.0395e-11,-1.0
"""ENSG00000140443""","""IGF1R""","""arm_fatfree_mass_right_int""",-0.013644,82469,-0.412619,2.8609e-8,-1.0
…,…,…,…,…,…,…,…
"""ENSG00000112077""","""RHAG""","""reticulocyte_count_int""",0.033239,9849,0.952411,1.8038e-41,1.0
"""ENSG00000029534""","""ANK1""","""reticulocyte_count_int""",0.02685,54620,0.656849,2.1682e-10,1.0
"""ENSG00000197969""","""VPS13A""","""reticulocyte_count_int""",0.003971,55477,0.176348,2.1188e-7,1.0


In [61]:
gene_trait_df['region'].n_unique(), gene_trait_df.shape[0]

(352, 1176)

In [62]:
or_threshold_anno = 0.95
or_threshold_pheno = 0.99

# --- 1. Lazily prepare the filter keys ---
region_keys = gene_trait_df.lazy().select(pl.col('region').unique())
anno_ids_lazy = melted_anno.lazy().join(
    region_keys, on='region', how='semi'
).select(pl.col('id').unique())


# --- Build main query (same as before, but stop before group_by) ---
gp_lazy = (
    appv
    .join(anno_ids_lazy, on="id", how="semi")
    .join(
        melted_anno.lazy().drop([c for c in melted_anno.columns if 'is_nan' in c]), 
        on="id", 
        how="inner"
    )
    .join(
        gene_trait_df.lazy(), 
        on=["region", "phenotype"], 
        how="inner"
    )
    .with_columns(
        mean_pheno_value_dircor_ptile = pl.when(pl.col('corr_dir') == -1)
            .then(1 - pl.col('mean_pheno_value_ptile'))
            .otherwise(pl.col('mean_pheno_value_ptile')),
    )
    
    .group_by(["annotation", "region", "gene_name", "phenotype"])
    .agg(
        n_dis_above_cutoff = (
            (pl.col("annotation_score_dircor") == 1) & 
            (pl.col("mean_pheno_value_dircor_ptile") >= or_threshold_pheno)
        ).sum(),
        n_notdis_above_cutoff = (
            (pl.col("annotation_score_dircor") == 1) & 
            (pl.col("mean_pheno_value_dircor_ptile") < or_threshold_pheno)
        ).sum(),
        n_dis_below_cutoff = (
            (pl.col("annotation_score_dircor") == 0) & 
            (pl.col("mean_pheno_value_dircor_ptile") >= or_threshold_pheno)
        ).sum(),
        n_notdis_below_cutoff = (
            (pl.col("annotation_score_dircor") == 0) & 
            (pl.col("mean_pheno_value_dircor_ptile") < or_threshold_pheno)
        ).sum()
    )
    .with_columns(
        # odds_ratio = ((pl.col("n_dis_above_cutoff") + 1) / (pl.col("n_notdis_above_cutoff")+1)) / (1 - or_threshold_pheno)
        odds_ratio = (pl.col("n_dis_above_cutoff") / pl.col("n_notdis_above_cutoff")) / (pl.col("n_dis_below_cutoff") / pl.col("n_notdis_below_cutoff"))
    )
)

# Execute
print("Executing with odds ratios...")
or_df = gp_lazy.collect(engine='streaming')
or_df

Executing with odds ratios...


annotation,region,gene_name,phenotype,n_dis_above_cutoff,n_notdis_above_cutoff,n_dis_below_cutoff,n_notdis_below_cutoff,odds_ratio
str,str,str,str,u64,u64,u64,u64,f64
"""am_pathogenicity""","""ENSG00000081479""","""LRP2""","""cystatin_c_int""",15,276,34,1875,2.997123
"""loftee_hc""","""ENSG00000080345""","""RIF1""","""forced_expiratory_volume_in_1s…",0,60,14,1575,0.0
"""am_pathogenicity""","""ENSG00000137198""","""GMPR""","""high_light_scatter_reticulocyt…",2,71,2,101,1.422535
"""am_pathogenicity""","""ENSG00000152270""","""PDE3B""","""high_light_scatter_reticulocyt…",0,84,5,444,0.0
"""am_pathogenicity""","""ENSG00000214706""","""IFRD2""","""immature_reticulocyte_fraction…",0,0,0,0,NaN
…,…,…,…,…,…,…,…,…
"""loftee_hc""","""ENSG00000087237""","""CETP""","""hdl_cholesterol_int""",3,38,6,342,4.5
"""am_pathogenicity""","""ENSG00000140443""","""IGF1R""","""trunk_predicted_mass_int""",14,137,14,476,3.474453
"""loftee_hc""","""ENSG00000111700""","""SLCO1B3""","""total_bilirubin_int""",4,59,7,499,4.83293


In [63]:
(    
    or_df
    .filter(
        (pl.col('annotation') == 'loftee_hc') &
        (pl.col("odds_ratio").is_finite())
    )
    .drop_nans()
)

annotation,region,gene_name,phenotype,n_dis_above_cutoff,n_notdis_above_cutoff,n_dis_below_cutoff,n_notdis_below_cutoff,odds_ratio
str,str,str,str,u64,u64,u64,u64,f64
"""loftee_hc""","""ENSG00000080345""","""RIF1""","""forced_expiratory_volume_in_1s…",0,60,14,1575,0.0
"""loftee_hc""","""ENSG00000152270""","""PDE3B""","""apolipoprotein_b_int""",0,70,15,784,0.0
"""loftee_hc""","""ENSG00000169047""","""IRS1""","""forced_vital_capacity_fvc_best…",1,44,5,896,4.072727
"""loftee_hc""","""ENSG00000110243""","""APOA5""","""platelet_distribution_width_in…",0,21,4,230,0.0
"""loftee_hc""","""ENSG00000138688""","""KIAA1109""","""whole_body_fat_mass_int""",5,253,40,2557,1.26334
…,…,…,…,…,…,…,…,…
"""loftee_hc""","""ENSG00000244734""","""HBB""","""haematocrit_percentage_int""",3,12,1,100,25.0
"""loftee_hc""","""ENSG00000087237""","""CETP""","""hdl_cholesterol_int""",3,38,6,342,4.5
"""loftee_hc""","""ENSG00000111700""","""SLCO1B3""","""total_bilirubin_int""",4,59,7,499,4.83293


In [64]:
plt_df = (
    or_df
    .drop_nans()
    .filter(pl.col("odds_ratio").is_finite())
    .with_columns(
        n_gene_phenos = pl.len().over('annotation'),
        med_odds_ratio = pl.col('odds_ratio').median().over('annotation'),
        avg_odds_ratio = pl.col('odds_ratio').mean().over('annotation'),
        std_odds_ratio = pl.col('odds_ratio').std().over('annotation'),
    )
    .with_columns(
        std_ci_upper = pl.col('avg_odds_ratio') + 1.96 * (pl.col('std_odds_ratio')),
        std_ci_lower = pl.col('avg_odds_ratio') - 1.96 * (pl.col('std_odds_ratio')),
    )
    .select(['annotation', 'n_gene_phenos', 'med_odds_ratio', 'avg_odds_ratio', 'std_odds_ratio', 'std_ci_upper', 'std_ci_lower'])
    .unique()
)
plt_df

annotation,n_gene_phenos,med_odds_ratio,avg_odds_ratio,std_odds_ratio,std_ci_upper,std_ci_lower
str,u64,f64,f64,f64,f64,f64
"""loftee_hc""",1161,3.670588,8.690164,19.707309,47.31649,-29.936162
"""am_pathogenicity""",1055,1.932039,3.1922,4.670953,12.347267,-5.962867


In [65]:
melted_anno.group_by('annotation').agg(
    n_vars = pl.col('annotation_score').filter(pl.col('annotation_score') > 0).count()
).sort('n_vars', descending=True)

annotation,n_vars
str,u64
"""am_pathogenicity""",57424
"""loftee_hc""",43640
